In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import networkx as nx

from qlinks.caging import (
    CageClassificationConfig,
    CageSearchConfig,
    CageSearcher,
    classify_cage_state,
)
from qlinks.models import (
    SquareQLMModel,
    SquareQDMModel,
    TriangularQLMModel,
    TriangularQDMModel,
    HoneycombQLMModel,
    HoneycombQDMModel,
)
from qlinks.visualizer import (
    LinkVisualStyle,
    BasisGridVisualizer,
    HamiltonianGraphStyle,
    HamiltonianGraphVisualizer,
)

# Model definition

In [ ]:
model = SquareQLMModel.from_qdm_staggered_background(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    # charge_magnitude=1,
    kinetic=1.0,
    potential=1.0,
)
print(model.allowed_sector_labels())

In [ ]:
build_result = model.build(
    basis_solver="dfs",
    builder="sparse",
    backend="scipy",
    sort_basis=True,
    on_missing="raise",
)

hamiltonian_matrix = build_result.hamiltonian
kinetic_matrix = build_result.kinetic
potential_matrix = build_result.potential
basis = build_result.basis

print("n_states =", basis.n_states)
print("H shape =", hamiltonian_matrix.shape)
print("H nnz =", hamiltonian_matrix.nnz)
print("K nnz =", kinetic_matrix.nnz)
print("V nnz =", potential_matrix.nnz)
print("K is bipartite =", nx.is_bipartite(nx.from_scipy_sparse_array(kinetic_matrix, edge_attribute="weight")))

# Search for caged states

In [ ]:
searcher = CageSearcher.from_model_build_result(
    build_result,
    config=CageSearchConfig(
        search_type="qlm",
        # type1_kappas=(0,),
        # type2_kappas=(-2, 2),
        tolerance=1e-10,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=256,
        ipr_candidate_count=128,
        ipr_random_seed=1234,
    ),
)

search_result = searcher.run()
print("number of cages:", len(search_result))
print("signatures:", search_result.signatures)
print("counts by signature:", search_result.counts_by_signature)

In [ ]:
signature = (0, 4)
record_index = -1
record = search_result[signature, record_index]

print("selected signature:", record.signature)
print("support size:", record.support.size)
print("energy:", record.cage_state.energy)

## Classify the selected caged state

In [ ]:
report = classify_cage_state(
    record.cage_state,
    kinetic_matrix=build_result.kinetic,
    basis_configs=basis.states,
    hilbert_size=search_result.hilbert_size,
    config=CageClassificationConfig(
        amplitude_tolerance=1e-10,
        cancellation_tolerance=1e-9,
        action_tolerance=1e-9,
    ),
)

print(report)

# Visualize the caged state on Hamiltonian graph

In [ ]:
graph_visualizer = HamiltonianGraphVisualizer.from_sparse_matrix(
    kinetic_matrix,
    include_self_loops=False,
    style=HamiltonianGraphStyle(
        cmap="coolwarm",
        label_vertices=True,
        vertex_size=18,
    ),
)

graph_visualizer.plot(
    backend="igraph-mpl",
    color_by="state_amplitude_real",
    state_vector=record.full_state,
    layout="kk",
    title=f"Fock-space graph colored by cage-state signed amplitude",
)

### Save the graph (optional)

In [ ]:
graph_visualizer.save_graph(
    f"../data/fock_graph.gexf",
    layout_backend="igraph",
    layout="kk",
    color_by="state_amplitude_real",
    state_vector=record.full_state,
)

# Visualize the basis states for
* caged state
* interference zeros
* all basis states (optional)

In [ ]:
grid_visualizer = BasisGridVisualizer(
    lattice=model.lattice,
    layout=model.layout,
    periodic_image_mode="positive_patch",
    collapse_duplicate_visual_links=True,
    site_label_style="sublattice_cell",
    # coordinate_transform=np.array(
    #     [
    #         [1.0, 0.0],
    #         [0.0, 0.72],
    #     ],
    #     dtype=float,
    # ),
    # style=LinkVisualStyle(
    #     node_size=50.0,
    #     plaquette_symbol_fontsize=12,
    #     plaquette_symbol_offset=(-0.01, 0.01),
    # ),
)

In [ ]:
grid_visualizer.plot_cage_support(
    search_result,
    basis_configs=basis.states,
    signature=signature,
    record_index=record_index,
    mode="arrows",
    plaquette_symbols="square_qlm",
    show_config_label=False,
)

In [ ]:
grid_visualizer.plot_interference_zeros(
    report,
    basis_configs=basis.states,
    mechanism="all",
    mode="arrows",
    plaquette_symbols="square_qlm",
    show_config_label=False,
)

In [ ]:
grid_visualizer.plot(
    basis.states,
    ncols=4,
    mode="arrows",
    plaquette_symbols="square_qlm",
    show_config_label=False,
    # suptitle="All basis configurations",
    single_plot_kwargs={
        "with_site_labels": False,
        "with_site_values": False,
        "with_link_values": False,
    },
)